# Setup Dataset e Preparazione per ResNet

## 1. Import e configurazioni di setup

In [1]:
import os
import sys
import glob # modulo per trovare file che corrispondono a un pattern (esempio: *.jpeg)
from pathlib import Path # per gestire meglio i file al posto delle stringhe
import time  # per tracciare i tempi di training

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plot
import seaborn as sns
# per visualizzare i grafici direttamente nel notebook
%matplotlib inline 

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader # classe che carica i dati in batch per il training
from torchvision import transforms, models

from tqdm import tqdm # mi serve così visualizzo meglio l'avanzamento quando questo è troppo lungo

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = Path('..')
BATCH_SIZE = 32 
IMG_SIZE = 224
N_CLASSES = 5  # Apple, Banana, Grape, Mango, Strawberry
# dimensioni standard

## 2. Normalizzazione e trasformazioni
Servono a 

In [2]:
NORMALIZE = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

TRAIN_TRANSFORMS = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),  # Augmentation automatica
    transforms.ToTensor(),
    NORMALIZE
])

VAL_TEST_TRANSFORMS = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    NORMALIZE
])

## 3. Verifica Dataset Scaricato

Il dataset è già pre-diviso in:
- **train/** (~1900-1940 immagini per classe)
- **valid/** (~39-40 immagini per classe)
- **test/** (~19-20 immagini per classe)

Totale: ~2000 immagini per classe (perfettamente bilanciato)

In [3]:
dataset_path = ROOT / "data" / "fruits-classification"

if dataset_path.exists():
    print(f"Dataset trovato in: {dataset_path}\n")
    
    # Conta le immagini per classe in ogni split
    class_data = {}
    
    for split in ['train', 'valid', 'test']:
        split_path = dataset_path / split
        print(f"{split.upper()}:")
        
        for class_name in sorted(os.listdir(split_path)):
            class_folder = split_path / class_name
            if class_folder.is_dir():
                n_images = len(list(class_folder.glob("*.jpeg")))
                
                if class_name not in class_data:
                    class_data[class_name] = {}
                class_data[class_name][split] = n_images
                print(f"  {class_name}: {n_images} immagini")
    
    # Stampa il totale per ogni classe
    print("\nTOTALE PER CLASSE:")
    for class_name in sorted(class_data.keys()): #keys è un metodo che restituisce le chiavi di un dizionario
        total = sum(class_data[class_name].values())
        print(f"  {class_name}: {total} immagini")
    
    print("\nDataset bilanciato")
else:
    print(f"Dataset non trovato in {dataset_path}")

Dataset trovato in: ../data/fruits-classification

TRAIN:
  Apple: 1923 immagini
  Banana: 1935 immagini
  Grape: 1940 immagini
  Mango: 1940 immagini
  Strawberry: 1940 immagini
VALID:
  Apple: 39 immagini
  Banana: 40 immagini
  Grape: 40 immagini
  Mango: 40 immagini
  Strawberry: 40 immagini
TEST:
  Apple: 19 immagini
  Banana: 20 immagini
  Grape: 20 immagini
  Mango: 20 immagini
  Strawberry: 20 immagini

TOTALE PER CLASSE:
  Apple: 1981 immagini
  Banana: 1995 immagini
  Grape: 2000 immagini
  Mango: 2000 immagini
  Strawberry: 2000 immagini

Dataset bilanciato


## 4. Verifica stratificazione
Verifico che il dataset sia stratificato nelle percentuali 80%, 10%, 10%

In [4]:

# Conta le immagini in ogni split
split_counts = {}

for split in ['train', 'valid', 'test']:
    split_path = dataset_path / split
    n_images = len(list(split_path.glob("*/*.jpeg")))
    split_counts[split] = n_images

total_images = sum(split_counts.values())
train_perc = (split_counts['train'] / total_images) * 100
valid_perc = (split_counts['valid'] / total_images) * 100
test_perc = (split_counts['test'] / total_images) * 100

print(f"Train: {split_counts['train']} immagini ({train_perc:.1f}%)")
print(f"Valid: {split_counts['valid']} immagini ({valid_perc:.1f}%)")
print(f"Test: {split_counts['test']} immagini ({test_perc:.1f}%)")
print(f"Total: {total_images} immagini\n")

# Se è già 80/10/10, va bene cosi, altrimenti redistribuisco
# La verifica la faccio con una tolleranza dell'1% (standard per dataset di piccole dimensioni)
if 79 <= train_perc <= 81 and 9 <= valid_perc <= 11 and 9 <= test_perc <= 11:
    print("Dataset stratificato: 80%, 10%, 10%")
else:
    print("Dataset non stratificato: eseguo ridistribuzione")
    
    # Leggi tutti i percorsi e le label da tutte le immagini
    all_paths = list(dataset_path.glob("*/*/*.jpeg"))  # attenzione ai livelli delle cartelle
    all_classes = [p.parent.name for p in all_paths]  
    
    # Primo split 80/20
    train_paths, temp_paths, train_classes, temp_classes = train_test_split(
        all_paths, all_classes, test_size=0.2, random_state=RANDOM_SEED, stratify=all_classes
    )
    
    # Secondo split 20/20 = 10% e 10%
    valid_paths, test_paths, valid_classes, test_classes = train_test_split(
        temp_paths, temp_classes, test_size=0.5, random_state=RANDOM_SEED, stratify=temp_classes
    )
    
    print(f"  Train: {len(train_paths)} immagini ({len(train_paths)/total_images*100:.1f}%)")
    print(f"  Valid: {len(valid_paths)} immagini ({len(valid_paths)/total_images*100:.1f}%)")
    print(f"  Test: {len(test_paths)} immagini ({len(test_paths)/total_images*100:.1f}%)")
    
    # Salva il dataset redistribuito
    import shutil
    new_dataset_path = ROOT / "data" / "fruits-classification-stratified"
    new_dataset_path.mkdir(parents=True, exist_ok=True)
    
    for split_name, paths_list, labels_list in [
        ('train', train_paths, train_classes),
        ('valid', valid_paths, valid_classes),
        ('test', test_paths, test_classes)
    ]:
        split_dir = new_dataset_path / split_name
        split_dir.mkdir(exist_ok=True)
        
        for img_path, class_name in zip(paths_list, labels_list):
            class_dir = split_dir / class_name
            class_dir.mkdir(exist_ok=True)
            shutil.copy(str(img_path), str(class_dir / img_path.name))
    
    dataset_path = new_dataset_path

Train: 9678 immagini (97.0%)
Valid: 199 immagini (2.0%)
Test: 99 immagini (1.0%)
Total: 9976 immagini

Dataset non stratificato: eseguo ridistribuzione
  Train: 7980 immagini (80.0%)
  Valid: 998 immagini (10.0%)
  Test: 998 immagini (10.0%)


## 5. DataLoader 
### Preparazione dei Dati per la Rete Neurale

I DataLoader sono strumenti che si occupano di:

1. Creazione di batch: Le immagini non vengono caricate tutte insieme, ma suddivise in pacchetti (batch) per evitare problemi di memoria
2. Mescolamento: Le immagini vengono mescolate casualmente durante il training, così la rete neurale apprende realmente e non impara solo l'ordine (così evito l'overfitting), metre niente mescolamento per validation e test (mi servono immagini pulite e risproducibilità)
3. Trasformazioni: Conversione delle immagini in tensori matematici, normalizzazione, augmentation (già definite nella cella 2)

In [5]:
from torchvision.datasets import ImageFolder

# Crea i dataset direttamente dalle cartelle (molto piu comodo)
train_dataset = ImageFolder(root=dataset_path / 'train', transform=TRAIN_TRANSFORMS)
val_dataset = ImageFolder(root=dataset_path / 'valid', transform=VAL_TEST_TRANSFORMS)
test_dataset = ImageFolder(root=dataset_path / 'test', transform=VAL_TEST_TRANSFORMS)

# Crea i DataLoader con specifiche di batch size, shuffle e num_workers standard
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print("Dataset e DataLoader creati")
print(f"Train: {len(train_dataset)} immagini, {len(train_loader)} batch")
print(f"Validation: {len(val_dataset)} immagini, {len(val_loader)} batch")
print(f"Test: {len(test_dataset)} immagini, {len(test_loader)} batch")

Dataset e DataLoader creati
Train: 7980 immagini, 250 batch
Validation: 998 immagini, 32 batch
Test: 998 immagini, 32 batch


# ResNet18 e Fine Tuning

## 1. Setup e Import di ResNet

### 1.1. Importazione ResNet18

**ResNet18** è una rete neurale convoluzionale (CNN) già addestrata su 1 milione di immagini (dataset ImageNet), ha 18 layer (strati), pesa circa 45MB e normalmente classifica 1000 categorie (perché ImageNet ne ha 1000)

In [6]:
# Importazione di ResNet18 da torchvision
# pretrained=True scarica i pesi da ImageNet (45MB la prima volta, poi cache locale)
model = models.resnet18(pretrained=True)

print("ResNet18 importato e caricato con pesi pre-addestrati")

# sposto il modello dalla RAM al device (GPU o CPU)
model = model.to(device)

print(f"Modello spostato su: {device}")

ResNet18 importato e caricato con pesi pre-addestrati
Modello spostato su: cuda


/home/imane/Thesis_AML/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/imane/Thesis_AML/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


### 1.2. Architettura ResNet18

**Struttura:**
- Layer convoluzionali (estraggono features dalle immagini)
- Layer batch normalization
- Layer ReLU (attivazione)
- Ultimo layer FC (fully connected) con 1000 output

L'ultimo layer verrà poi cambiato da 1000 a 5 (le classi del dataset)

In [7]:
print("Struttura di ResNet18:")
print(model)
print("\nImportante: output_features=1000")

Struttura di ResNet18:
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=

### 1.3. Parametri del Modello

I **parametri** sono i pesi della rete neurale
Ogni neurone ha pesi che si aggiustano durante l'allenamento
(ResNet18 circa 11.7 milioni di parametri)

In [8]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Parametri totali: {total_params:,}")
print(f"Parametri addestrabili: {trainable_params:,}")

Parametri totali: 11,689,512
Parametri addestrabili: 11,689,512


## 2. Adattamento del modello

### 2.1. Cambiare l'ultimo layer FC (Fully Connected)

L'ultimo layer di ResNet18 ha 1000 output (le classi di ImageNet) mentre al progetto ne servono solo 5 (Apple, Banana, Grape, Mango, Strawberry)

Quindi bisogna togliere il layer FC vecchio e mettere un nuovo layer FC con input --> 512 (rimane uguale, è il numero di feature estratte), output --> 5 (le 5 classi)

In [16]:
# Prima: Linear(in_features=512, out_features=1000)
# Dopo: Linear(in_features=512, out_features=5)

model.fc = nn.Linear(in_features=512, out_features=N_CLASSES).to(device)

print(f"  Nuovo output: {model.fc.out_features} classi (invece di 1000)")
print(f"\nModello adattato per il dataset corrente")

  Nuovo output: 5 classi (invece di 1000)

Modello adattato per il dataset corrente


### 2.2. Congelare i layer precedenti

Il freeze dei layer precedenti fa si che i pesi di questi non cabmbino dureante l'allenamento
Questo serve per non allenare la rete da zero

Dalla struttura del modello so che ci sono 4+1 layers e dato che ImageNet è molto diverso dal mio dataset lascio addestrabile anche il quarto layer oltre che al FC(strategia moderata)

Si lasciano congelati i primi tre strati
Il quarto e l'ultimo layer si lasciano addestrabili (finetuning)

In [10]:
for param in model.layer1.parameters():
    param.requires_grad = False 
    # il gradiente sui pesi dei paraemntri di questo layer non vengono calcolati
    # quindi non vengono aggiornati e rimangono fissi

for param in model.layer2.parameters():
    param.requires_grad = False

for param in model.layer3.parameters():
    param.requires_grad = False

print("Layer 1, 2, 3 congelati (non si addestrano)")
print("Layer 4 e FC layer rimangono trainabili")

# verifica dei parametri trainabili
trainable_params_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parametri totali: {total_params:,}")
print(f"\nParametri trainabili: {trainable_params_after:,} (prima erano {trainable_params:,})")

Layer 1, 2, 3 congelati (non si addestrano)
Layer 4 e FC layer rimangono trainabili
Parametri totali: 11,689,512

Parametri trainabili: 8,405,829 (prima erano 11,689,512)


### 2.3. Funzione di perdita e funzione di ottimizzazione

Una funzione di perdita misura quanto il modello sbaglia su una predizione (più alta la loss, più grande l'errore)

La funzione di ottimizzazione usa la loss per aggiornare i pesi e ridurre l'errore

Scelte fatte:
- **CrossEntropyLoss**: per classificazione multi-classe
- **Adam**: il migliore per il finetuning

In [11]:
# Loss Function
loss_fn = nn.CrossEntropyLoss()
# Optimizer
learning_rate = 0.0001
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print("Loss Function: CrossEntropyLoss")
print(f"Optimizer: Adam")
print(f"Learning Rate: {learning_rate}")

print("\nFinetuning completato")

Loss Function: CrossEntropyLoss
Optimizer: Adam
Learning Rate: 0.0001

Finetuning completato


## 3. Training Loop

Il training loop è la parte più importante dell'addestramento della rete neurale, in pratica è il processo per aggiornare i pesi della rete usando i dati di training per minimizzare la loss

### 3.1. Funzione di training

La funzione di training esegue un'epoca completa:
1. Passa tutti i dati di training attraverso il modello (forward pass)
2. Calcola la loss con CrossEntropyLoss
3. Calcola i gradienti (backward pass)
4. Aggiorna i pesi con l'optimizer
5. Torna all'inizio e ripete per il prossimo batch

In [12]:
def train_one_epoch(model, train_loader, loss_fn, optimizer, device):
    """
    Esegue un'epoca di training
    
    Ritorna: (avg_loss, accuracy)
    """
    model.train()  # modalita training
    
    tot_loss = 0
    correct_preds = 0
    tot_preds = 0
    
    for images, labels in tqdm(train_loader, desc="Training"):   # itera su tutti i batch di training con barra di progresso
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass: calcolo delle predizioni
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        
        # Backward pass: calcolo dei gradienti
        optimizer.zero_grad()  # pulisce i gradienti vecchi
        loss.backward()  # calcola nuovi gradienti
        optimizer.step()  # aggiorna i pesi
        
        tot_loss += loss.item()
        predicted = torch.argmax(outputs, dim=1)  # prendo la classe con score massimo
        correct_preds += (predicted == labels).sum().item()
        tot_preds += labels.size(0)
    
    avg_loss = tot_loss / len(train_loader)
    accuracy = correct_preds / tot_preds
    
    return avg_loss, accuracy

print("Funzione train_one_epoch() definita")

Funzione train_one_epoch() definita


### 3.2. Funzione di validazione

La funzione di validazione è simile al training, ma NON aggiorna i pesi (niente backward pass, niente optimizer.step())

Calcola solo loss e accuracy per capire se il modello sta imparando bene e viene usata dopo ogni epoca per decidere se salvare il modello

In [18]:
def val_one_epoch(model, val_loader, loss_fn, device):
    """
    Esegue un'epoca di validazione
    
    Ritorna: (avg_loss, accuracy)
    """
    model.eval()  # mette il modello in modalità evaluation (no dropout, no batch norm updates)
    
    tot_loss = 0
    correct_preds = 0
    tot_preds = 0
    
    # torch.no_grad() dice a PyTorch di non calcolare i gradienti (più veloce)
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation"):
            images = images.to(device)
            labels = labels.to(device)
            
            # Forward pass: calcola le predizioni
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            
            tot_loss += loss.item()
            predicted = torch.argmax(outputs, dim=1)
            correct_preds += (predicted == labels).sum().item()
            tot_preds += labels.size(0)
    
    avg_loss = tot_loss / len(val_loader)
    accuracy = correct_preds / tot_preds
    
    return avg_loss, accuracy

print("Funzione val_one_epoch() definita")

Funzione val_one_epoch() definita


### 3.3. Parametri del training

Prima di iniziare il training, definiamo:
- **num_epochs**: numero di volte che la rete vede tutti i dati
- **best_val_loss**: tracciare la miglior validation loss per salvare il modello
- **metrics**: dizionario per salvare le metriche di ogni epoca

In [14]:
n_epochs = 5  # numero di epoche
# inizializzo ad infinito per garantire che alla prima epoca il modello venga salvato come best model
best_val_loss = float('inf')  
model_save_path = ROOT / "models" / "best_resnet18.pt"
model_save_path.parent.mkdir(parents=True, exist_ok=True)

# Early Stopping: ferma il training se non c'è miglioramento
patience = 2  # numero di epoche senza miglioramento prima di fermarsi
patience_counter = 0

# dizionario per tracciare le metriche
metrics = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'epoch_time': []
}

print(f"Numero di epoche: {n_epochs}")
print(f"Early Stopping patience: {patience} epoche")

Numero di epoche: 5
Early Stopping patience: 2 epoche


### 3.4. Loop principale di Training

Esecuzione completa del training con Early Stopping:
1. Per ogni epoca: **train_one_epoch()** → **validate_one_epoch()**
2. Salvataggio del modello migliore basato su validation loss
3. Arresto anticipato se nessun miglioramento per `patience` epoche
4. Riepilogo finale con tutte le metriche e statistiche di timing

In [ ]:
# Training loop principale con Early Stopping
start_time_tot = time.time()  # inizio timing totale

for epoch in range(n_epochs):
    start_time_epoch = time.time()  # inizio timing per epoca
        
    # Training
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    
    # Validazione
    val_loss, val_acc = val_one_epoch(model, val_loader, loss_fn, device)
    
    # calcola tempo per epoca
    epoch_time = time.time() - start_time_epoch
    
    metrics['train_loss'].append(train_loss)
    metrics['train_acc'].append(train_acc)
    metrics['val_loss'].append(val_loss)
    metrics['val_acc'].append(val_acc)
    metrics['epoch_time'].append(epoch_time)
    
    # Early Stopping: controlla se c'è miglioramento
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), model_save_path)
        patience_counter = 0  # reset counter
        print(f"Nuovo migliore modello salvato (Val Loss: {val_loss:.4f})\n")
    else:
        patience_counter += 1
        print("Nessun miglioramento\n")
    
    # Early Stopping: ferma se nessun miglioramento per 'patience' epoche
    if patience_counter >= patience:
        print(f"Early Stopping! Nessun miglioramento per {patience} epoche consecutive")
        print(f"Training fermato alla epoca {epoch+1}")
        break


print("\nTRAINING COMPLETATO")

# calcolo tempo totale formattato
tot_time = time.time() - start_time_tot
hours = int(tot_time // 3600)
minutes = int((tot_time % 3600) // 60)
seconds = int(tot_time % 60)
tot_time_formatted = f"{hours:02d}:{minutes:02d}:{seconds:02d}"

# DataFrame per le metriche
metrics_df = pd.DataFrame(metrics)
metrics_df.index.name = 'Epoca'
metrics_df.index = metrics_df.index+1 

print("\nMetriche:")
print(metrics_df.to_string())

print(f"\nTempi di training:")
print(f"  Tempo totale: {tot_time_formatted}")
print(f"  Tempo per epoca (media): {np.mean(metrics['epoch_time']):.2f}s")

Training:   0%|          | 0/250 [00:00<?, ?it/s]

Validation: 100%|██████████| 32/32 [00:00<00:00, 94.60it/s] 


Nuovo migliore modello salvato (Val Loss: 0.3316)



Validation: 100%|██████████| 32/32 [00:00<00:00, 95.93it/s] 


Nuovo migliore modello salvato (Val Loss: 0.2775)



...

Nuovo migliore modello salvato (Val Loss: 0.2580)



Validation: 100%|██████████| 32/32 [00:00<00:00, 92.66it/s]


Nessun miglioramento



Validation: 100%|██████████| 32/32 [00:00<00:00, 90.46it/s] 

Nessun miglioramento

Early Stopping! Nessun miglioramento per 2 epoche consecutive
Training fermato alla epoca 5

TRAINING COMPLETATO

Metriche:
       train_loss  train_acc  val_loss   val_acc  epoch_time
Epoca                                                       
1        0.623507   0.774687  0.331602  0.879760    3.250369
2        0.376191   0.867419  0.277534  0.902806    3.005540
3        0.298087   0.896742  0.257985  0.917836    3.068575
4        0.234602   0.920927  0.266894  0.914830    3.112374
5        0.210286   0.928697  0.265958  0.905812    3.113551

Tempi di training:
  Tempo totale: 00:00:15
  Tempo per epoca (media): 3.11s


## 4. Valutazione sul Test Set

In [ ]:
# Carica il miglior modello salvato
model.load_state_dict(torch.load(model_save_path))
print(f"Miglior modello caricato")

# Valuta sul test set usando la stessa funzione di validazione
test_loss, test_acc = validate_one_epoch(model, test_loader, loss_fn, device)


In [ ]:
print("\nRISULTATI SUL TEST SET")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")


In [ ]:
# Ottieni i nomi delle classi
class_names = test_dataset.classes

print("\nDataset loaded successfully")


## 5. Visualizzazioni

Visualizziamo i risultati del training con grafici e heatmap


In [ ]:
# Curve di training
metrics_plot = pd.DataFrame(metrics)
epochs = range(1, len(metrics_plot) + 1)

fig, (ax1, ax2) = plot.subplots(1, 2, figsize=(13, 5))

ax1.plot(epochs, metrics['train_loss'], 'o-', label='Train')
ax1.plot(epochs, metrics['val_loss'], 's-', label='Validation')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, metrics['train_acc'], 'o-', label='Train')
ax2.plot(epochs, metrics['val_acc'], 's-', label='Validation')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plot.tight_layout()
plot.show()


In [ ]:
# Visualizza esempi di predizioni dal test set
fig, axes = plot.subplots(2, 5, figsize=(14, 6))
axes = axes.flatten()

model.eval()
with torch.no_grad():
    images, labels = next(iter(test_loader))
    images = images.to(device)
    labels = labels.to(device)
    outputs = model(images)
    predicted = torch.argmax(outputs, dim=1)
    
    for i in range(10):
        img = images[i].cpu().numpy()
        img = np.transpose(img, (1, 2, 0))
        img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img = np.clip(img, 0, 1)
        
        axes[i].imshow(img)
        true_label = class_names[labels[i].item()]
        pred_label = class_names[predicted[i].item()]
        color = 'green' if true_label == pred_label else 'red'
        axes[i].set_title(f"{true_label}\n→{pred_label}", color=color, fontsize=9)
        axes[i].axis('off')

plot.tight_layout()
plot.show()


## 6. Riassunto Finale

Progetto completato! Qui sotto il riassunto completo dell'addestramento e valutazione.


In [ ]:
# Riassunto finale
best_epoch = np.argmin(metrics['val_loss'])

print("\nRIASSUNTO FINALE")
print(f"  Modello: ResNet18 (pretrained)")
print(f"  Dataset: Fruits (5 classi)")
print(f"  Learning Rate: {learning_rate}")
print(f"  Max Epochs: {n_epochs}, Early Stopping Patience: {patience}")

print(f"\n  Train Loss (finale): {metrics['train_loss'][-1]:.4f}")
print(f"  Train Accuracy (finale): {metrics['train_acc'][-1]:.4f}")
print(f"  Val Loss (migliore): {metrics['val_loss'][best_epoch]:.4f}")
print(f"  Val Accuracy (migliore): {metrics['val_acc'][best_epoch]:.4f}")

print(f"\n  Test Loss: {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

print(f"\n  Tempo totale: {tot_time_formatted}")
print(f"  Tempo medio per epoca: {np.mean(metrics['epoch_time']):.2f}s")
print(f"  Modello salvato in: {model_save_path}")

print("\n✅ Progetto completato!\n")
